# Phase 3: LLM Verification & Ingredient Intelligence Engine

In this phase, we use **Groq + Llama 3.3 70B** as our LLM Agent to:
1. **Semantic Verification**: Verify true cross-border product matches and filter out false positives.
2. **E-Number Decoding**: Automatically decode food additive codes (e.g. INS 330 → Citric Acid).
3. **Ingredient & Health Intelligence**: Compare Indian vs International formulations for hidden additives.

In [5]:
# Cell 1: Setup - Install Groq library and configure API key

%pip install -q groq

from groq import Groq

# Paste your Groq API key here (starts with gsk_...)
GROQ_API_KEY = "gsk_your_groq_api_key_here"

# Connect Python to Groq's ultra-fast LPU inference engine
client = Groq(api_key=GROQ_API_KEY)

# Quick test: send a simple message to Llama 3.3 70B
test_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say: GROQ CONNECTED!"}]
)

print("✅ Groq API Connected!")
print("🤖 Model Response:", test_response.choices[0].message.content)


Note: you may need to restart the kernel to use updated packages.
✅ Groq API Connected!
🤖 Model Response: GROQ CONNECTED!


In [6]:
# Cell 2: Load Enriched Dataset & Pick Product #1 (7 UP)

import json

# 1. Open the Phase 2 output file
with open('data/benchmark_candidates.json', 'r', encoding='utf-8') as f:
    all_products = json.load(f)

# 2. Filter for matched items only
matched_products = [p for p in all_products if p.get('OFF_UK_Match_Found_Y_N') == 'Yes']

# 3. Pick the very first item (7 UP) to test on
item1 = matched_products[0]

print(f"Total matched items found: {len(matched_products)}")
print("Item Name:", item1['Item name'])
print("Indian ING:", item1['Ingredients'])
print("Global ING:", item1['OFF_UK_Ingredients'])


Total matched items found: 151
Item Name: 7 Up Lemon Soft Drink
Indian ING: Carbonated Water, Sugar, Acidity Regulators (330,Acidity Regulators 331,Acidity Regulators 296), Preservative (211)
Global ING: carbonated water, sugar, acids (citric acid, malic acid), natural lemon and lime flavouring with other natural flavourings, acidity regulator (sodium citrate), sweetner (steviol glycosides),


In [7]:
# Cell 3: Ask Llama (via Groq) to analyze item #1 (7 UP) in plain text

prompt = f"""
You are an expert FMCG Food Scientist.

Analyze this product:
Item Name: {item1['Item name']}
Brand: {item1['Brand_Name']}

Indian Ingredients: {item1['Ingredients']}
Global Ingredients: {item1['OFF_UK_Ingredients']}

Tasks:
1. Is this a valid product match? (YES or NO)
2. Decode Indian E-numbers (e.g. 330 -> Citric Acid, 211 -> Sodium Benzoate).
3. Give a 2-sentence summary of differences between the Indian vs Global recipe.
"""

# Send to Groq (Llama 3.3 70B)
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

# Print Llama's answer
print(response.choices[0].message.content)


1. NO
2. Indian E-numbers: 
   - 330: Citric Acid
   - 331: Sodium Citrate
   - 296: Malic Acid
   - 211: Sodium Benzoate
3. The Indian recipe for 7 Up Lemon Soft Drink differs from the global recipe in that it uses sugar and preservative sodium benzoate, whereas the global recipe uses steviol glycosides as a sweetener and does not explicitly list sodium benzoate as a preservative. Additionally, the global recipe mentions natural lemon and lime flavoring with other natural flavorings, which is not explicitly stated in the Indian ingredients list.


In [8]:
# Cell 4: Bulletproof Structured JSON from Groq (JSON Mode)

import json

prompt = f"""
You are an FMCG Food Scientist. Analyze this product match and return ONLY valid JSON.

Product: {item1['Item name']}
Brand: {item1['Brand_Name']}
Indian Ingredients: {item1['Ingredients']}
Global Ingredients: {item1['OFF_UK_Ingredients']}

Return a JSON object with EXACTLY these keys:
- is_valid_match: true or false (boolean)
- match_confidence: integer 0 to 100
- decoded_e_numbers: object mapping each E-number found to its chemical name
- key_difference: one sentence comparing Indian vs Global recipe
"""

# Groq JSON Mode: Forces Llama to return ONLY pure JSON
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    response_format={"type": "json_object"}   # ← Pure JSON mode, same as Gemini!
)

# Parse JSON result
result = json.loads(response.choices[0].message.content)

# Print it nicely
print("✅ Valid Match?  :", result['is_valid_match'])
print("📊 Confidence   :", result['match_confidence'], "%")
print("🧪 E-Numbers    :", result['decoded_e_numbers'])
print("📝 Key Diff     :", result['key_difference'])


✅ Valid Match?  : True
📊 Confidence   : 80 %
🧪 E-Numbers    : {'211': 'Sodium Benzoate', '296': 'Maleic acid', '330': 'Citric acid', '331': 'Sodium citrate'}
📝 Key Diff     : The Indian version of 7 Up Lemon Soft Drink uses Sugar, whereas the Global version uses Sugar and an additional sweetener Steviol glycosides.


In [9]:
# Cell 5: Full Batch Loop - Analyze ALL 151 Products with Groq!
# Groq allows 30 requests/minute on free tier = we can use time.sleep(2) only!

import time
import json

results = []

print(f"🚀 Starting AI Analysis for {len(matched_products)} products via Groq...\n")

for i, product in enumerate(matched_products):
    print(f"[{i+1}/{len(matched_products)}] Analyzing: {product['Item name'][:40]}...", end=" ")
    
    prompt = f"""
    You are an FMCG Food Scientist. Analyze this product match and return ONLY valid JSON.
    
    Product: {product['Item name']}
    Brand: {product['Brand_Name']}
    Indian Ingredients: {product['Ingredients']}
    Global Ingredients: {product['OFF_UK_Ingredients']}
    
    Return JSON with EXACTLY these keys:
    - is_valid_match: true or false
    - match_confidence: integer 0 to 100
    - decoded_e_numbers: object mapping E-numbers to chemical names
    - key_difference: one sentence summary comparing Indian vs Global formula
    """
    
    # Smart Retry Loop: if rate limit hit, wait 10s and retry automatically
    while True:
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"}
            )
            result = json.loads(response.choices[0].message.content)
            result['item_name'] = product['Item name']
            result['brand'] = product['Brand_Name']
            results.append(result)
            print("✅")
            break  # Success! Move to next product
            
        except Exception as e:
            err_msg = str(e)
            if "429" in err_msg or "rate" in err_msg.lower():
                print("⏳ Rate limit hit! Sleeping 10s...", end=" ")
                time.sleep(10)
            else:
                print(f"❌ Error: {e}")
                results.append({
                    "item_name": product['Item name'],
                    "brand": product['Brand_Name'],
                    "is_valid_match": False
                })
                break
    
    # 2-second sleep: Keeps us safely under Groq's 30 RPM free tier limit
    time.sleep(2)

# Save final results to disk
with open('data/llm_ingredient_intelligence.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=4)

print("\n" + "="*55)
print(f"🎉 BATCH AI ANALYSIS COMPLETE!")
print(f"✅ Successfully analyzed: {sum(1 for r in results if r.get('is_valid_match') != False)} / {len(results)} products")
print(f"💾 Saved to: data/llm_ingredient_intelligence.json")
print("="*55)


🚀 Starting AI Analysis for 151 products via Groq...

[1/151] Analyzing: 7 Up Lemon Soft Drink... ✅
[2/151] Analyzing: 7UP Nimbooz Soft Drink... ✅
[3/151] Analyzing: Betty Crocker Triple Chocolate Brownie I... ✅
[4/151] Analyzing: Cadbury Bournville Cranberry 50% Dark CH... ✅
[5/151] Analyzing: Cadbury Bournville Fruit & Nut 50% Dark ... ✅
[6/151] Analyzing: Cadbury Bournville Rich Cocoa 70% Dark C... ✅
[7/151] Analyzing: Cadbury Temptations Almond Treat Premium... ✅
[8/151] Analyzing: Cadbury Temptations Rum & Raisins Premiu... ✅
[9/151] Analyzing: Bournvita Chocolate... ✅
[10/151] Analyzing: Cadbury 5 Star 3D CHOCOLATE BAR... ✅
[11/151] Analyzing: Cadbury 5 Star CHOCOLATE BAR... ✅
[12/151] Analyzing: Cadbury Bournville Rich Cocoa 50% Dark C... ✅
[13/151] Analyzing: Cadbury Dairy Milk Bites Hazelnut Chocol... ✅
[14/151] Analyzing: Cadbury Dairy Milk Chocolate Bar... ✅
[15/151] Analyzing: Cadbury Dairy Milk Crispello CHOCOLATE B... ✅
[16/151] Analyzing: Cadbury Dairy Milk Fruit & Nut CH

In [10]:
# Cell 6: Confidence Tier Breakdown

import json
import pandas as pd

with open('data/llm_ingredient_intelligence.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_llm = pd.DataFrame(data)

strict    = (df_llm['is_valid_match'] == True).sum()
tier2     = (df_llm['match_confidence'] >= 60).sum()
tier3     = (df_llm['match_confidence'] >= 40).sum()

print("=" * 50)
print(f"🥇 Tier 1 - Strict Exact Twins (True + High Conf)  : {strict}")
print(f"🥈 Tier 2 - Strong Brand Matches  (Conf >= 60%)     : {tier2}")
print(f"🥉 Tier 3 - Usable Candidates     (Conf >= 40%)     : {tier3}")
print(f"📦 Total Analyzed                                   : {len(df_llm)}")
print("=" * 50)


🥇 Tier 1 - Strict Exact Twins (True + High Conf)  : 47
🥈 Tier 2 - Strong Brand Matches  (Conf >= 60%)     : 126
🥉 Tier 3 - Usable Candidates     (Conf >= 40%)     : 132
📦 Total Analyzed                                   : 151
